In [5]:
!pip install sqlalchemy

!pip install pymysql

%pip install cryptography --upgrade

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sqlalchemy import create_engine


In [7]:
# 2. rentals_month function

def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    """
    Retrieves rental data for a given month and year from the Sakila database.
    
    Parameters:
        engine: SQLAlchemy engine object for Sakila DB
        month: integer (1-12), target month
        year: integer (e.g., 2005), target year
    
    Returns:
        DataFrame containing rental_id, rental_date, inventory_id, customer_id, staff_id
    """
    query = f"""
        SELECT rental_id, rental_date, inventory_id, customer_id, staff_id
        FROM rental
        WHERE MONTH(rental_date) = {month}
        AND YEAR(rental_date) = {year};
    """
    return pd.read_sql(query, engine)

In [8]:
# 3. rental_count_month function

def rental_count_month(rental_df: pd.DataFrame, month: int, year: int) -> pd.DataFrame:
    """
    Counts rentals by customer for a given month/year.
    
    Parameters:
        rental_df: DataFrame returned by rentals_month
        month: integer (1-12)
        year: integer (e.g., 2005)
    
    Returns:
        DataFrame with customer_id and rentals_{month}_{year}
    """
    col_name = f"rentals_{str(month).zfill(2)}_{year}"
    
    counts = (
        rental_df.groupby("customer_id")["rental_id"]
        .count()
        .reset_index()
        .rename(columns={"rental_id": col_name})
    )
    
    return counts

In [9]:
# 4. compare_rentals function

def compare_rentals(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    """
    Compares rentals between two months and adds a difference column.
    
    Parameters:
        df1, df2: DataFrames returned by rental_count_month
    
    Returns:
        Combined DataFrame with both months’ rental counts and difference column
    """
    merged = pd.merge(df1, df2, on="customer_id", how="inner")
    
    # Difference = rentals_month2 - rentals_month1
    col1, col2 = merged.columns[1], merged.columns[2]
    merged["difference"] = merged[col2] - merged[col1]
    
    return merged
